# Recipe Generator with Specific Constraints

Create a prompt that generates recipes with the following constraints:
- Must use exactly 5 ingredients (no more, no less)
- Must be cooked in 30 minutes or less
- Must be suitable for beginners
- Should include a creative title

Test your prompt with at least two different cuisine types.

In [ ]:
# pip install -U langchain-openai

Installo **langchain-openai per evitare gli errori:
```
WARNING! top_p is not default parameter.
    e predict deprecato
```

In [7]:
# Import delle librerie
import os
from dotenv import load_dotenv
from langchain.llms import OpenAI
# from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI # nuovo import
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from typing import List
from pydantic import BaseModel, Field

In [8]:
# carico la chiave nel mio .env
load_dotenv()

True

## Inizializzo il modello 

In [12]:
nerd_chat = ChatOpenAI(
    model="gpt-3.5-turbo", 
    temperature=0.7,  # Controls randomness: lower is more deterministic
    max_tokens=None,
    timeout=None,
    top_p=None,
    n=1,
    max_retries=2,
)

# Let's test our model
response = chat.invoke("Briefly explain what a python dev can do") # uso invoke invece di predict
print(response.content) # agggiungo dot notation per prendere solo il content della risposta

A Python developer can write code using the Python programming language to create web applications, software applications, data analysis tools, machine learning algorithms, automation scripts, and more. They can work on a wide range of projects across various industries, from web development to data science to artificial intelligence. Python developers are responsible for designing, developing, testing, and maintaining code to ensure that it meets the requirements of the project and functions properly.


## Recipe Generation
### Templating
userò nel templating le istruzioni fornite sopra. E ci aggiungo un pò di mondo nerd e vediamo che succede 🤓

In [22]:
recipe_template = PromptTemplate(
    input_variables=['ingredients', 'cuisine','dietary_restrictions'],
    template="""
    Create a detailed recipe picking max 5 of these given ingredients:{ingredients}.
    The recupe should be {cuisine} style and consider these dietary restrictions: {dietary_restrictions}.
    The total cooking time(from ingredients preparations, to finished product) can't be more than 30 minutes.
    The difficulty must be suistable for beginners.
    Include servings, ingredients with measurements, and step-by-step instructions.
    Also include a creative title for the recipes, for that you can use reference from character of the game magic the gathering.
    The final users are 80% nerd people!
    """
)

# Creo la chain ovvero prompt più modello
recipe_chain = LLMChain(llm=nerd_chat, prompt=recipe_template)

# Generate a recipe
recipe = recipe_chain.run({
    'ingredients': 'salmon, rise, wakame, onion, olive, pizza, ananas',
    'cuisine': "Japanese",
    'dietary_restrictions':"low carb"
})

print(recipe)

Title: Jace's Low Carb Japanese Salmon Bowl

Servings: 2

Ingredients:
- 2 salmon fillets
- 1 cup of cooked brown rice
- 1/2 cup of wakame seaweed
- 1/2 onion, thinly sliced
- 1 tablespoon of olive oil
- Salt and pepper to taste

Instructions:
1. Preheat the oven to 400°F (200°C).
2. Place the salmon fillets on a baking sheet lined with parchment paper. Drizzle with olive oil and season with salt and pepper.
3. Bake the salmon in the preheated oven for 15-20 minutes, or until cooked through.
4. In a medium saucepan, heat the olive oil over medium heat. Add the sliced onion and cook until softened.
5. Add the cooked brown rice and wakame seaweed to the saucepan and stir to combine. Cook for another 2-3 minutes until heated through.
6. To assemble the bowls, divide the rice mixture between two bowls. Top each bowl with a baked salmon fillet.
7. Serve hot and enjoy your delicious and healthy Japanese-style low carb salmon bowl!

Enjoy your meal, nerds!


## Advanced Prompt 
### Uso SystemMessagePromptTemplate and HHumanMessagePromptTemplate

In [23]:
# Create a system message that provides context and instruction
system_template = """You are a professional chef with expertise in {cuisine} cuisine. 
You are a nerd passionate about the Magic The Gathering.
Your task is to create delicious recipes that accommodate {dietary_restrictions} dietary needs.
Be creative but practical, using ingredients that are commonly available. You are talking with begginers."""

system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# Creo un messaggio HumanMessagePromptTemplate

# Create a human message that provides the specific request
human_template = """Generate a detailed recipe picking max 5 of these ingredients: {ingredients}.
Please include:
1. Creative recipe title
2. Total preparation and cooking time
3. Difficulty level (Easy, Medium, Hard)
4. Number of servings
5. Ingredients with precise measurements
6. Step-by-step cooking instructions
7. Nutritional highlights
8. Serving suggestions"""

human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# Combine the messages into a chat prompt template
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# Create the chain
advanced_recipe_chain = LLMChain(llm=nerd_chat, prompt=chat_prompt)

# Generate a recipe
advanced_recipe = advanced_recipe_chain.run({
    "cuisine": "Italian",
    "dietary_restrictions": "vegetarian",
    "ingredients": "pasta, tomatoes, basil, garlic, olive oil, mozzarella"
})

print(advanced_recipe)

**Recipe Title:** Caprese Pasta Delight

**Total Preparation and Cooking Time:** 30 minutes

**Difficulty Level:** Easy

**Number of Servings:** 4

**Ingredients:**
- 12 oz pasta (such as spaghetti or penne)
- 2 cups cherry tomatoes, halved
- 1/2 cup fresh basil leaves, chopped
- 4 cloves garlic, minced
- 1/4 cup olive oil
- 8 oz fresh mozzarella, diced
- Salt and pepper to taste

**Step-by-Step Cooking Instructions:**
1. Cook the pasta according to package instructions until al dente. Drain and set aside.
2. In a large pan, heat olive oil over medium heat. Add minced garlic and cook for 1-2 minutes until fragrant.
3. Add the cherry tomatoes to the pan and cook for another 3-4 minutes until they start to soften.
4. Toss in the cooked pasta and chopped basil. Mix well to combine all the flavors.
5. Gently fold in the diced mozzarella until it starts to melt slightly.
6. Season with salt and pepper to taste.
7. Serve hot and garnish with extra basil leaves and a drizzle of olive oil.

**

# Creating a Weekly Meal Planner

In [25]:
# Create a system message for the meal planner
meal_planner_system = """You are a professional nutritionist and meal planning expert.
Your task is to create a balanced, healthy weekly meal plan that fits the user's dietary preferences,
restrictions, calorie goal.
Each day should include breakfast, lunch, dinner, and a snack."""

meal_planner_system_prompt = SystemMessagePromptTemplate.from_template(meal_planner_system)

# Create a human message for the specific meal plan request
meal_planner_human = """Create a 7-day meal plan with the following criteria:
- Dietary preference: {diet_type}
- Allergies/restrictions: {restrictions}
- Daily calorie target: {calorie_target}
- Cooking complexity preference: {cooking_complexity}
- Number of people: {people_count}


For each day, provide:
1. Day of the week, adding a creative title
2. Breakfast (with brief recipe)
3. Lunch (with brief recipe)
4. Dinner (with brief recipe)
5. Snack option
6. Estimated calorie count for each meal
7. A grocery list for the entire week at the end"""

meal_planner_human_prompt = HumanMessagePromptTemplate.from_template(meal_planner_human)

# Combine into a chat prompt template
meal_planner_prompt = ChatPromptTemplate.from_messages([meal_planner_system_prompt, meal_planner_human_prompt])

# Create the chain
meal_planner_chain = LLMChain(llm=chat, prompt=meal_planner_prompt)

# Generate a meal plan
meal_plan = meal_planner_chain.run({
    "diet_type": "Japanese",
    "restrictions": "no shellfish, low sodium",
    "calorie_target": "2000",
    "cooking_complexity": "moderate, with some quick options for busy days",
    "people_count": "2"
})

print(meal_plan)

**Weekly Japanese-Inspired Meal Plan**

**Day 1: Umami Breakfast Delight**
- **Breakfast:** Miso Soup with Tofu and Wakame: Start your day with a warm and comforting bowl of miso soup filled with tofu and seaweed.
- **Lunch:** Teriyaki Chicken Rice Bowl: Grilled chicken glazed with a homemade teriyaki sauce served over steamed rice and a side of stir-fried vegetables.
- **Dinner:** Salmon Teriyaki with Soba Noodles: Pan-seared salmon fillets with a flavorful teriyaki glaze, served with buckwheat soba noodles and steamed broccoli.
- **Snack:** Edamame: Enjoy a handful of steamed edamame pods sprinkled with a touch of sea salt.
- **Calories:** Breakfast (250), Lunch (600), Dinner (550), Snack (100)

**Day 2: Sushi Lover's Dream**
- **Breakfast:** Tamago Sando (Japanese Egg Sandwich): Fluffy scrambled eggs sandwiched between two slices of soft white bread.
- **Lunch:** California Roll Salad: Deconstructed California roll with imitation crab, avocado, cucumber, and mixed greens drizzled wi

## Structured Output with LangChain + Pydantic

In [28]:
# Define a Pydantic model for a recipe with the new constraints
from pydantic import BaseModel, Field, field_validator
from typing import List, ClassVar

class Ingredient(BaseModel):
    name: str = Field(description="The name of the ingredient")
    quantity: str = Field(description="The quantity of the ingredient needed")
    unit: str = Field(description="The unit of measurement for the ingredient")

class RecipeStep(BaseModel):
    step_number: int = Field(description="The step number in the cooking process")
    instruction: str = Field(description="The detailed cooking instruction for this step")

class Recipe(BaseModel):
    title: str = Field(description="A creative title for the recipe")
    cooking_time: str = Field(description="Total time required to prepare and cook (must be 30 minutes or less)")
    difficulty: str = Field(description="Difficulty level (must be suitable for beginners)")
    servings: int = Field(description="Number of servings this recipe makes")
    ingredients: List[Ingredient] = Field(description="List of exactly 5 ingredients with quantities (no more, no less)")
    instructions: List[RecipeStep] = Field(description="Step-by-step cooking instructions")
    nutritional_info: str = Field(description="Brief nutritional highlights")
    serving_suggestion: str = Field(description="Suggestion on how to serve the dish")
    
    # Validator to ensure exactly 5 ingredients
    @field_validator('ingredients')
    @classmethod
    def validate_ingredients_count(cls, v):
        if len(v) != 5:
            raise ValueError('Recipe must contain exactly 5 ingredients (no more, no less)')
        return v
    
    # Validator to ensure cooking time is 30 minutes or less
    @field_validator('cooking_time')
    @classmethod
    def validate_cooking_time(cls, v):
        if "hour" in v.lower() or "hr" in v.lower():
            raise ValueError('Cooking time must be 30 minutes or less')
        try:
            # Extract numeric part of cooking time
            import re
            minutes = re.findall(r'\d+', v)
            if minutes and int(minutes[0]) > 30:
                raise ValueError('Cooking time must be 30 minutes or less')
        except:
            pass  # If we can't parse the time, we'll let it pass and rely on the prompt
        return v
    
    # Validator to ensure the recipe is suitable for beginners
    @field_validator('difficulty')
    @classmethod
    def validate_difficulty(cls, v):
        if v.lower() not in ['easy', 'beginner', 'simple']:
            raise ValueError('Recipe must be suitable for beginners')
        return v

# Create a parser for the recipe
from langchain.output_parsers import PydanticOutputParser
recipe_parser = PydanticOutputParser(pydantic_object=Recipe)

# Create a format instruction
format_instructions = recipe_parser.get_format_instructions()

# Create a structured recipe prompt with the new constraints
from langchain.prompts import PromptTemplate
structured_recipe_template = """You are a professional chef who specializes in creating delicious recipes that are quick and easy to make.

Create a {cuisine} recipe with the following constraints:
- Must use EXACTLY 5 ingredients (no more, no less)
- Must be cooked in 30 minutes or less
- Must be suitable for beginners
- Should include a creative title

Based on these requirements, create a recipe using these ingredients: {ingredients}.
Consider these dietary restrictions: {dietary_restrictions}.

{format_instructions}
"""

structured_prompt = PromptTemplate(
    template=structured_recipe_template,
    input_variables=["cuisine", "ingredients", "dietary_restrictions"],
    partial_variables={"format_instructions": format_instructions}
)

# Create a chain for structured output
from langchain.chains import LLMChain
structured_recipe_chain = LLMChain(llm=chat, prompt=structured_prompt)

# Test with Italian cuisine
italian_output = structured_recipe_chain.run({
    "cuisine": "Italian",
    "ingredients": "pasta, tomatoes, garlic, basil, olive oil",
    "dietary_restrictions": "vegetarian"
})
print("Italian Recipe:")
print(italian_output)
print("\n" + "-"*50 + "\n")

# Test with Mexican cuisine
mexican_output = structured_recipe_chain.run({
    "cuisine": "Mexican",
    "ingredients": "tortillas, beans, avocado, lime, cilantro",
    "dietary_restrictions": "dairy-free"
})
print("Mexican Recipe:")
print(mexican_output)

Italian Recipe:
{
  "title": "Easy Caprese Pasta",
  "cooking_time": "30 minutes",
  "difficulty": "Beginner",
  "servings": 4,
  "ingredients": [
    {
      "name": "pasta",
      "quantity": "8 oz",
      "unit": "uncooked"
    },
    {
      "name": "tomatoes",
      "quantity": "2",
      "unit": "medium"
    },
    {
      "name": "garlic",
      "quantity": "2 cloves",
      "unit": ""
    },
    {
      "name": "basil",
      "quantity": "1/4 cup",
      "unit": "chopped"
    },
    {
      "name": "olive oil",
      "quantity": "2 tbsp",
      "unit": ""
    }
  ],
  "instructions": [
    {
      "step_number": 1,
      "instruction": "Cook pasta according to package instructions until al dente. Drain and set aside."
    },
    {
      "step_number": 2,
      "instruction": "In a large skillet, heat olive oil over medium heat. Add minced garlic and cook until fragrant."
    },
    {
      "step_number": 3,
      "instruction": "Add diced tomatoes to the skillet and cook for 5-